# Notebook 4: Graph analytics

Import the Python libraries.

In [24]:
import re

import kuzu
import networkx as nx
import polars as pl
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [16]:
%load_ext watermark
%watermark
%watermark --iversions

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Last updated: 2025-11-11T23:37:30.165316-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

networkx : 3.4.2
kuzu     : 0.9.0
watermark: 2.5.0
polars   : 1.29.0



## Graph analytics in NetworkX

Reconnect to the existing graph database.

In [6]:
DB_PATH: str = "./db"

db: kuzu.Database = kuzu.Database(DB_PATH)
conn: kuzu.Connection = kuzu.Connection(db)

In [6]:
data_path: pathlib.Path = pathlib.Path("data")

We'll load a slice of the [OpenSanctions](https://www.opensanctions.org/) dataset, which provides the "risk" category of data.
This describes people and organizations who represent known risks for FinCrime.

In [7]:
df1 = pl.read_ndjson(data_path / "open-sanctions.json")
df1.head(3)

DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,GENDER,RISKS,ADDRESSES,DATES,COUNTRIES,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,CONTACTS
str,str,str,str,list[struct[3]],str,list[struct[1]],list[struct[7]],list[struct[2]],list[struct[3]],list[struct[9]],list[struct[1]],list[struct[5]],str,list[struct[1]]
"""OPEN-SANCTIONS""","""NK-25vyVFzt8vdJGgAXMRTwTJ""","""PERSON""","""2024-07-30T16:41:14""","[{""PRIMARY"",null,""Abassin BADSHAH""}]",null,"[{""corp.disqual""}]","[{""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",null,null,null,null,null,null}]","[{null,""1985-05-12""}]","[{null,""gb"",null}]","[{null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-25vyVFzt8vdJGgAXMRTwTJ""}]","[{""https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk""}]","[{""Directorship"",""OPEN-SANCTIONS"",""NK-SKAADAiqiZ78JsJjeg72Te"",null,null}, {""Directorship"",""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE"",null,null}]","""https://www.opensanctions.org/…",null
"""OPEN-SANCTIONS""","""NK-3p3mmVWmjwVtTfKchz4kNE""","""ORGANIZATION""","""2025-01-07T00:33:03""","[{""PRIMARY"",""LMAR (GB) LTD"",null}]",null,null,"[{""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",null,null,null,null,null,""BUSINESS""}]",null,"[{""gb"",null,null}]","[{null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE""}]",null,"[{null,null,null,""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE""}]","""https://www.opensanctions.org/…",null
"""OPEN-SANCTIONS""","""NK-auyPsLrBzRoxjCRWgjBvas""","""ORGANIZATION""","""2024-03-03T19:51:29""","[{""PRIMARY"",""WANDLE HOLDINGS LIMITED"",null}]",null,"[{""sanction.linked""}]","[{""DEANA BEACH APTS, BLOCK A, Flat 212, Προμαχών Ελευθερίας, 33, 'Αγιος Αθανάσιος, 4103, Λεμεσός, Κύπρος"",null,null,null,null,null,""BUSINESS""}]","[{""2006-12-08"",null}]","[{""cy"",null,null}]","[{""C188266"",null,null,null,null,null,null,null,null}, {""HE188266"",null,null,null,null,null,null,null,null}, {null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-auyPsLrBzRoxjCRWgjBvas""}]",null,"[{null,null,null,""OPEN-SANCTIONS"",""NK-auyPsLrBzRoxjCRWgjBvas""}]","""https://opensanctions.org/enti…",null


Each entity ID from OpenSanctions has a risk classification. This can be useful to associate an ID with a particular risk, allowing us to narrow down on candidates that are relevant to a particular investigation.

In [8]:
# Get risks from OpenSanctions
df_risk = os.extract_risks(df1)
df_risk.head(3)

id,topic
str,str
"""NK-25vyVFzt8vdJGgAXMRTwTJ""","""corp.disqual"""
"""NK-auyPsLrBzRoxjCRWgjBvas""","""sanction.linked"""
"""NK-cf4Q3KcmUnQbt8Cy7iTtwK""","""sanction.linked"""


We're now ready to extract the open sanctions data. The `extract_open_sanctions` function will take the raw data, process the nested fields within it and return the relevant columns that we need for our graph.

In [9]:
df_os = os.extract_open_sanctions(df1)
df_os.head(3)

id,kind,name,addr,url
str,str,str,str,str
"""NK-25vyVFzt8vdJGgAXMRTwTJ""","""PERSON""","""Abassin BADSHAH""","""31 Quernmore Close, Bromley, K…","""https://www.opensanctions.org/…"
"""NK-3p3mmVWmjwVtTfKchz4kNE""","""ORGANIZATION""","""LMAR (GB) LTD""","""31 Quernmore Close, Bromley, K…","""https://www.opensanctions.org/…"
"""NK-L2UmsZtsyvYiaEmHSaiZ2t""","""PERSON""","""Gulnara Suleimanova KERIMOVA""","""MOSCOW, RUS, 123430""","""https://www.opensanctions.org/…"


Of particular interest for this workshop is the person ["Abassin Badshah"](https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk), former owner of multiple Papa John's franchises in London, who is disqualified from being a corporate director until 2026, due to his [tax evasion conviction](https://londonnewsonline.co.uk/news/catford-papa-johns-pizza-boss-jailed-after-669000-tax-evasion/) in 2021.

[Open Ownership](https://www.openownership.org/) describes _ultimate beneficial ownership_ (UBO) details, which provides the "link" category of data. In other words, "Who owns how much of what, and who actually has controlling interest?"

In [10]:
df2 = pl.read_ndjson(data_path / "open-ownership.json")
df2.head(3)

DATA_SOURCE,RECORD_ID,statementDate,RECORD_TYPE,NAMES,PRIMARY_NAME_FULL,personType,ATTRIBUTES,ADDRESSES,IDENTIFIERS,LINKS,RELATIONSHIPS,replaces_statements,REGISTRATION_DATE,dissolutionDate,REGISTRATION_COUNTRY,DATE_OF_BIRTH
str,str,str,str,list[struct[2]],str,str,list[struct[1]],list[struct[3]],list[struct[3]],list[struct[3]],list[struct[7]],list[struct[1]],str,str,str,str
"""OPEN-OWNERSHIP""","""10094521532396971848""","""2023-06-18""","""ORGANIZATION""","[{""GOLD WYNN UK HOLDINGS LIMITED"",null}]",null,null,null,"[{""BUSINESS"",""C/O Fladgate Llp, 16 Great Queen Street, London, WC2B 5DG"",""GB""}]","[{""12524623"",""GB-COH"",""GBR""}]","[{""https://opencorporates.com/companies/gb/12524623"",null,null}, {null,""https://register.openownership.org/entities/18432059995972240708"",null}]","[{""OOR"",""10094521532396971848"",null,null,null,null,null}, {null,null,""OOR"",""7584591804488095167"",""shareholding 75% 100%"",""2020-03-18"",""2020-04-29""}, … {null,null,""OOR"",""7584591804488095167"",""appointment_of_board"",""2020-03-18"",""2020-04-29""}]",null,"""2020-03-18""",null,"""GB""",null
"""OPEN-OWNERSHIP""","""10165632722354515453""","""2023-06-18""","""ORGANIZATION""","[{""UPSIDE TECHNOLOGY LIMITED"",null}]",null,null,null,"[{""BUSINESS"",""Apt 52, 3 Whitehall Court, London, SW1A 2EL"",""GB""}]","[{""12165794"",""GB-COH"",""GBR""}]","[{""https://opencorporates.com/companies/gb/12165794"",null,null}, {null,""https://register.openownership.org/entities/15659422647652524790"",null}]","[{""OOR"",""10165632722354515453"",null,null,null,null,null}, {null,null,""OOR"",""598161773989218568"",""shareholding 75% 100%"",""2019-08-20"",null}, … {null,null,""OOR"",""598161773989218568"",""appointment_of_board"",""2019-08-20"",null}]",null,"""2019-08-20""","""2022-10-11""","""GB""",null
"""OPEN-OWNERSHIP""","""10165632722354515453""","""2023-06-18""","""ORGANIZATION""","[{""UPSIDE TECHNOLOGY LIMITED"",null}]",null,null,null,"[{""BUSINESS"",""Apt 52, 3 Whitehall Court, London, SW1A 2EL"",""GB""}]","[{""12165794"",""GB-COH"",""GBR""}]","[{""https://opencorporates.com/companies/gb/12165794"",null,null}, {null,""https://register.openownership.org/entities/15659422647652524790"",null}]","[{""OOR"",""10165632722354515453"",null,null,null,null,null}, {null,null,""OOR"",""598161773989218568"",""shareholding 75% 100%"",""2019-08-20"",null}, … {null,null,""OOR"",""598161773989218568"",""appointment_of_board"",""2019-08-20"",null}]",null,"""2019-08-20""","""2022-10-11""","""GB""",null


Just like with the OpenSanctions data, we can use the `extract_open_ownership` function to process the nested JSON data and return the relevant columns that we need for our graph.

In [11]:
df_oo = oo.extract_open_ownership(df2)
df_oo.head(3)

id,kind,name,address,country
str,str,str,str,str
"""10094521532396971848""","""ORGANIZATION""","""GOLD WYNN UK HOLDINGS LIMITED""","""C/O Fladgate Llp, 16 Great Que…","""GB"""
"""10165632722354515453""","""ORGANIZATION""","""UPSIDE TECHNOLOGY LIMITED""","""Apt 52, 3 Whitehall Court, Lon…","""GB"""
"""10264459789712927869""","""PERSON""","""Kenneth Kurt Hansen""","""Finderupvej 61, Kastrup, 2770""","""DK"""


For the relationships in our graph, we'll need to select only the relationships that have **both** `src_id` and `dst_id` in the list of ids. This is done via the `extract_open_ownership_relationships` function.

In [12]:
ids = df_oo.select("id").to_series().to_list()
df_oa_relationships = oo.extract_open_ownership_relationships(df2, open_ownership_ids=ids)
df_oa_relationships.head(3)

src_id,dst_id,role,date
str,str,str,str
"""10094521532396971848""","""7584591804488095167""","""shareholding 75% 100%""","""2020-03-18"""
"""10094521532396971848""","""7584591804488095167""","""appointment_of_board""","""2020-03-18"""
"""10094521532396971848""","""7584591804488095167""","""voting_rights 75% 100%""","""2020-03-18"""


The final step to preprocess the data for our graph is to separate the entities by their source (whether they come from OpenSanctions or Open Ownership).

In [16]:
df_sz_oo = sz_export.df_rec.filter(pl.col("source") == "OPEN-OWNERSHIP").select("ent_id", "rec_id", "why", "level")
df_sz_oo.head(3)

ent_id,rec_id,why,level
str,str,str,i64
"""sz_1""","""17207853441353212969""","""+NAME+ADDRESS+NATIONALITY""",1
"""sz_1""","""6747548100436839873""","""+NAME+DOB+NATIONALITY""",1
"""sz_10""","""5927522753545014068""","""+NAME+DOB+NATIONALITY""",1


In [17]:
df_sz_os = sz_export.df_rec.filter(pl.col("source") == "OPEN-SANCTIONS").select("ent_id", "rec_id", "why", "level")
df_sz_os.head(3)

ent_id,rec_id,why,level
str,str,str,i64
"""sz_1""","""NK-25vyVFzt8vdJGgAXMRTwTJ""","""""",0
"""sz_2""","""NK-3p3mmVWmjwVtTfKchz4kNE""","""""",0
"""sz_3""","""NK-auyPsLrBzRoxjCRWgjBvas""","""""",0


There's a common design pattern used with investigative graphs:

 - Step 1: run _entity resolution_ to merge the structured data sources and generate graph elements
 - Step 2: partition the graph to _identify subgraphs_ as potential fraud networks -- typically with graph algorithms such as _Louvain partitioning_ or _strongly connected components_
 - Step 3: centrality to _rank individuals_ of interest within each subgraph -- to identify the most connected elements of a subgraph, as a most likely controlling party or ultimate beneficial owner (UBO)
 - Step 4: process these subgraphs through *case management* tools

Here we can use a Cypher query to approximate _connected components_.
We'll take this short-cut, extracting subgraphs of resolved entities and data records from OpenSanctions and Open Ownership, plus the relations among them.

We'll transfer to [`NetworkX`](https://networkx.org/) to analyze the subgraphs as a [`NetworkX.MultiDiGraph`](https://networkx.org/documentation/stable/reference/classes/multidigraph.html)

In [10]:
subgraphs: nx.MultiDiGraph = conn.execute("""
MATCH (a:Entity:OpenOwnership:OpenSanctions)-[b]->(c:Entity:OpenOwnership:OpenSanctions)
RETURN *
""").get_as_networkx(directed = True)

Using [_betweenness centrality_](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.centrality.betweenness_centrality.html) is a way to show important "bridge" nodes within a subgraph -- think of this as "focusing a lens" on a vicinity of the graph.
Then we'll store the results in a dataframe.

In [28]:
df_tween: pl.DataFrame = pl.DataFrame(
    {
        "id": re.sub(r"^\w+\_", "", id),
        "betweenness_centrality": rank,
    }
    for id, rank in nx.betweenness_centrality(subgraphs).items()
).sort("betweenness_centrality", descending = True)

df_tween.head(5)

id,betweenness_centrality
str,f64
"""sz:60""",0.001545
"""sz:249""",0.000887
"""sz:104""",0.000807
"""sz:47""",0.00077
"""sz:96""",0.000692


Add a new property `betweenness_centrality` as a new column to the relevant tables.

In [29]:
conn.execute("ALTER TABLE Entity ADD betweenness_centrality FLOAT");
conn.execute("ALTER TABLE OpenOwnership ADD betweenness_centrality FLOAT");
conn.execute("ALTER TABLE OpenSanctions ADD betweenness_centrality FLOAT");

In [31]:
conn.execute(
    f"""
    LOAD FROM df_tween
    MERGE (s1:Entity {{id: id}})
    SET s1.betweenness_centrality = betweenness_centrality
    MERGE (s2:OpenSanctions {{id: id}})
    SET s2.betweenness_centrality = betweenness_centrality
    MERGE (s3:OpenOwnership {{id: id}})
    SET s3.betweenness_centrality = betweenness_centrality
    """
);

Let's query to the see the top-ranked entities in a particular subgraph.

In [38]:
res = conn.execute("""
MATCH (c:Entity)-[b]->(a:Entity)
WHERE c.descrip CONTAINS "Abassin"
RETURN a.id, a.descrip, a.betweenness_centrality
ORDER BY a.betweenness_centrality DESC
LIMIT 10
""")

res.get_as_pl()

a.id,a.descrip,a.betweenness_centrality
str,str,f32
"""sz:155""","""BARLLOWS SERVICES LTD""",0.000017
"""sz:2""","""LMAR GB LTD""",0.000017
"""sz:9""","""BARLLOWS SERVICES LTD""",0.000017
"""sz:99""","""WELLHANCIA HEALTH CARE LTD""",0.000017
"""sz:156""","""Rehana Badshah""",0.000017
"""sz:ds_open-ownership_172078534…","""sz:ds_open-ownership:172078534…",0.0
"""sz:ds_open-ownership_674754810…","""sz:ds_open-ownership:674754810…",0.0
"""sz:ds_open-sanctions_NK-25vyVF…","""sz:ds_open-sanctions:NK-25vyVF…",0.0


Finally, close the database connection.

In [39]:
db.close()

---